# Notebook 7: Feature Importance in Random Forests

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2.5 hours | **Topic:** Tree-Based Methods & Gradient Boosting

---

## Why This Matters

Building a model that predicts house prices is only half the job. The other half — often more valuable to stakeholders — is understanding **which features drive prices**. 

Feature importance answers:
- Which renovations increase value the most?
- Should a buyer care more about school ratings or crime rates?
- Which data columns are worth expensive collection?

But feature importance methods disagree with each other, can be biased, and can be fooled. This notebook teaches you three methods, their failure modes, and when to trust each one.

---

## Real-World Analogy First

Imagine a panel of **500 appraisers**, each independently estimating house prices using 8 factors:

> **Location, Size (sqft), Age, School Rating, Crime Rate, Distance to downtown, Recent Renovations, Garage**

Each appraiser uses their own decision logic — some weight location heavily, others focus on size. After all estimates are made, you want to know: **which factor mattered most across all appraisers?**

**Method 1 — MDI (Mean Decrease in Impurity):** Ask each appraiser: "At each step of your reasoning, how much did splitting on *Location* vs *Size* vs ... reduce your uncertainty?" Average across all appraisers.

**Method 2 — Permutation Importance:** Take a new set of houses and randomly scramble all the *Location* values. How much worse do the appraisers get? Scrambling an important feature hurts a lot; scrambling a useless one barely matters.

**Method 3 — Drop-Column:** Remove one factor entirely. Retrain all appraisers. How much worse are they? This is the gold standard but requires retraining 500 appraisers × N features times.

## Concept: Mean Decrease in Impurity (MDI)

### Plain English
When a decision tree splits a node, it picks the feature and threshold that creates the most "pure" child nodes — groups where all houses have similar prices. MDI measures, for each feature, how much total impurity reduction it caused across all splits in all trees.

### Formula

For a single tree, the importance of feature $j$ at node $m$ is:

$$\text{MDI}_j^{\text{tree}} = \sum_{m: \text{split on } j} \frac{n_m}{n_{\text{total}}} \cdot \left[ I(m) - \frac{n_{m_L}}{n_m} I(m_L) - \frac{n_{m_R}}{n_m} I(m_R) \right]$$

Where:
- $n_m$ = number of samples at node $m$
- $I(m)$ = impurity at node $m$ (variance for regression, Gini for classification)
- $m_L, m_R$ = left and right children

Across the forest:
$$\text{MDI}(j) = \frac{1}{T} \sum_{t=1}^{T} \text{MDI}_j^{\text{tree}_t}$$

Then normalized so all importances sum to 1.

**Bias Warning:** MDI favors high-cardinality features (features with many unique values). A random float column can appear important simply because it offers many possible split thresholds — more chances to find a lucky split by chance.

---

## Concept: Permutation Importance

### Plain English
Break a feature by shuffling it randomly. If the model's accuracy drops a lot, the feature was important. If accuracy barely changes, the feature was not being used.

### Algorithm
1. Compute baseline score on validation set
2. For each feature $j$:
   - Shuffle column $j$ (breaking its relationship with target)
   - Measure new score
   - Importance = baseline − shuffled score
3. Repeat $n_{\text{repeats}}$ times, take mean

**Advantages:** Model-agnostic, unbiased toward high-cardinality features, uses validation set so reflects generalization.

**Caution with correlated features:** If features A and B are correlated, shuffling A while leaving B intact still lets the model "cheat" via B — both will appear less important than they truly are.

---

## Concept: Drop-Column Importance

Retrain the model without feature $j$. Measure performance drop.

$$\text{DropCol}(j) = R^2_{\text{full}} - R^2_{\text{without } j}$$

**Gold standard** — truly measures how much the feature contributes. **Expensive** — requires $n_{\text{features}} + 1$ model refits.

In [ ]:
# ── Cell 1: Imports and global settings ──────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully.")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

In [ ]:
# ── Cell 2: Generate synthetic house price dataset ───────────────────────────
# 800 samples, 10 features:
#   6 real features that genuinely influence price
#   4 noise features (2 binary, 1 low-cardinality int, 1 high-cardinality float)

np.random.seed(42)
n = 800

# ── Real features ──
location_score   = np.random.uniform(1, 10, n)      # 1=bad, 10=great neighborhood
sqft             = np.random.uniform(800, 4000, n)  # square footage
age_years        = np.random.uniform(0, 60, n)      # age of house
school_rating    = np.random.uniform(1, 10, n)      # local school quality
crime_rate       = np.random.uniform(0, 10, n)      # higher = more crime
distance_downtown= np.random.uniform(1, 30, n)      # miles to city center

# ── Noise features ──
noise_binary_1   = np.random.randint(0, 2, n).astype(float)  # binary noise
noise_binary_2   = np.random.randint(0, 2, n).astype(float)  # binary noise
noise_low_card   = np.random.randint(0, 5, n).astype(float)  # 5-level categorical noise
noise_float_hc   = np.random.uniform(0, 1000, n)             # HIGH-CARDINALITY noise float

# ── Price formula (only real features matter) ──
price = (
    120_000
    + 15_000 * location_score       # location is king
    + 80    * sqft                  # size matters a lot
    - 1_500 * age_years             # older houses cheaper
    + 8_000 * school_rating         # good schools add value
    - 6_000 * crime_rate            # crime hurts value
    - 3_000 * distance_downtown     # closer to city = more valuable
    + np.random.normal(0, 20_000, n)# realistic noise
)

# ── Assemble DataFrame ──
feature_names = [
    'location_score', 'sqft', 'age_years', 'school_rating',
    'crime_rate', 'distance_downtown',
    'noise_binary_1', 'noise_binary_2', 'noise_low_card', 'noise_float_hc'
]

real_features  = feature_names[:6]
noise_features = feature_names[6:]

X = np.column_stack([
    location_score, sqft, age_years, school_rating,
    crime_rate, distance_downtown,
    noise_binary_1, noise_binary_2, noise_low_card, noise_float_hc
])

df = pd.DataFrame(X, columns=feature_names)
df['price'] = price

X_train, X_val, y_train, y_val = train_test_split(X, price, test_size=0.25, random_state=42)

print(f"Dataset shape: {df.shape}")
print(f"Train: {X_train.shape[0]} samples | Val: {X_val.shape[0]} samples")
print(f"\nPrice range: ${price.min():,.0f} – ${price.max():,.0f}")
print(f"Price mean:  ${price.mean():,.0f}")
print(f"\nReal features:  {real_features}")
print(f"Noise features: {noise_features}")
df.describe().round(2)

In [ ]:
# ── Cell 3: Train Random Forest ───────────────────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_train = rf.predict(X_train)
y_pred_val   = rf.predict(X_val)

r2_train = r2_score(y_train, y_pred_train)
r2_val   = r2_score(y_val,   y_pred_val)
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))

print("Random Forest trained.")
print(f"  Train R²: {r2_train:.4f}")
print(f"  Val   R²: {r2_val:.4f}")
print(f"  Val RMSE: ${rmse_val:,.0f}")
print(f"\n  n_estimators : {rf.n_estimators}")
print(f"  n_features   : {rf.n_features_in_}")

In [ ]:
# ── Cell 4: MDI (Mean Decrease in Impurity) ───────────────────────────────────
mdi_importances = rf.feature_importances_
mdi_series = pd.Series(mdi_importances, index=feature_names).sort_values(ascending=False)

# Color: real features = teal, noise = coral
colors_mdi = ['#2a9d8f' if f in real_features else '#e76f51' for f in mdi_series.index]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(mdi_series.index[::-1], mdi_series.values[::-1], color=colors_mdi[::-1], edgecolor='white', height=0.7)

# Annotate values
for bar, val in zip(bars, mdi_series.values[::-1]):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

# Legend
legend_handles = [
    mpatches.Patch(color='#2a9d8f', label='Real feature'),
    mpatches.Patch(color='#e76f51', label='Noise feature'),
]
ax.legend(handles=legend_handles, loc='lower right')

ax.set_xlabel('MDI Importance (normalized)')
ax.set_title('MDI Feature Importance\n(notice: noise_float_hc appears significant!)', fontweight='bold')
ax.axvline(x=1/len(feature_names), color='gray', linestyle='--', alpha=0.6, label='Uniform baseline')
plt.tight_layout()
plt.show()

print("\nMDI Rankings:")
for rank, (feat, val) in enumerate(mdi_series.items(), 1):
    tag = '(REAL)  ' if feat in real_features else '(NOISE) '
    print(f"  {rank:2d}. {tag}{feat:<22} {val:.4f}")

In [ ]:
# ── Cell 5: Permutation Importance — from scratch ─────────────────────────────

def permutation_importance_scratch(model, X_val, y_val, metric_fn, n_repeats=10, seed=42):
    """
    Compute permutation importance for each feature.
    
    Returns:
        importances_mean : array of shape (n_features,)
            Mean importance across repeats (positive = feature is useful)
        importances_std  : array of shape (n_features,)
    """
    rng = np.random.RandomState(seed)
    baseline = metric_fn(y_val, model.predict(X_val))
    
    importances_matrix = np.zeros((X_val.shape[1], n_repeats))
    
    for j in range(X_val.shape[1]):
        for r in range(n_repeats):
            X_perm = X_val.copy()
            X_perm[:, j] = rng.permutation(X_perm[:, j])
            perm_score = metric_fn(y_val, model.predict(X_perm))
            # Positive = baseline was better = feature was useful
            importances_matrix[j, r] = baseline - perm_score
    
    return importances_matrix.mean(axis=1), importances_matrix.std(axis=1)


# Run scratch implementation
perm_mean_scratch, perm_std_scratch = permutation_importance_scratch(
    rf, X_val, y_val, metric_fn=r2_score, n_repeats=15
)

# Run sklearn's permutation_importance for comparison
perm_sklearn = permutation_importance(
    rf, X_val, y_val, n_repeats=15, random_state=42, n_jobs=-1, scoring='r2'
)

print("Scratch vs sklearn permutation importance (should be very close):")
print(f"{'Feature':<22} {'Scratch':>10} {'Sklearn':>10} {'Match?':>8}")
print("-" * 55)
for i, feat in enumerate(feature_names):
    s_val = perm_mean_scratch[i]
    k_val = perm_sklearn.importances_mean[i]
    match = "✓" if abs(s_val - k_val) < 0.01 else "✗ CHECK"
    print(f"{feat:<22} {s_val:>10.4f} {k_val:>10.4f} {match:>8}")

In [ ]:
# ── Cell 6: Visualize Permutation Importance ──────────────────────────────────
perm_series = pd.Series(perm_sklearn.importances_mean, index=feature_names).sort_values(ascending=False)
perm_std_sorted = pd.Series(perm_sklearn.importances_std, index=feature_names).reindex(perm_series.index)

colors_perm = ['#2a9d8f' if f in real_features else '#e76f51' for f in perm_series.index]

fig, ax = plt.subplots(figsize=(10, 5))
y_pos = range(len(perm_series))
ax.barh(
    list(perm_series.index)[::-1],
    list(perm_series.values)[::-1],
    xerr=list(perm_std_sorted.values)[::-1],
    color=colors_perm[::-1],
    edgecolor='white',
    height=0.7,
    capsize=4
)

ax.axvline(0, color='black', linewidth=0.8, linestyle='-')
ax.set_xlabel('Permutation Importance (drop in R²)')
ax.set_title('Permutation Feature Importance\n(error bars = std over 15 repeats)', fontweight='bold')

legend_handles = [
    mpatches.Patch(color='#2a9d8f', label='Real feature'),
    mpatches.Patch(color='#e76f51', label='Noise feature'),
]
ax.legend(handles=legend_handles, loc='lower right')
plt.tight_layout()
plt.show()

print("\nPermutation Rankings:")
for rank, (feat, val) in enumerate(perm_series.items(), 1):
    tag = '(REAL)  ' if feat in real_features else '(NOISE) '
    print(f"  {rank:2d}. {tag}{feat:<22} {val:+.4f}")

In [ ]:
# ── Cell 7: Drop-Column Importance ────────────────────────────────────────────
# Retrain RF without each feature, measure R² drop.
# This is the gold standard but requires n_features+1 model refits.

baseline_r2 = r2_score(y_val, rf.predict(X_val))
drop_col_importances = {}

print(f"Baseline R² (all features): {baseline_r2:.4f}\n")
print("Retraining without each feature...")

for j, feat in enumerate(feature_names):
    # Remove column j
    X_train_drop = np.delete(X_train, j, axis=1)
    X_val_drop   = np.delete(X_val,   j, axis=1)
    
    rf_drop = RandomForestRegressor(
        n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1
    )
    rf_drop.fit(X_train_drop, y_train)
    r2_drop = r2_score(y_val, rf_drop.predict(X_val_drop))
    drop_col_importances[feat] = baseline_r2 - r2_drop
    print(f"  Without {feat:<22}: R²={r2_drop:.4f}  drop={baseline_r2 - r2_drop:+.4f}")

drop_series = pd.Series(drop_col_importances).sort_values(ascending=False)
print(f"\nDone. {len(feature_names)+1} model fits completed.")

In [ ]:
# ── Cell 8: Side-by-Side Comparison of All Three Methods ─────────────────────
comparison_df = pd.DataFrame({
    'MDI':        mdi_series,
    'Permutation': perm_series,
    'Drop-Column': drop_series
}).fillna(0)

# Normalize all to [0, 1] for fair visual comparison
def normalize(s):
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else s

comparison_norm = comparison_df.apply(normalize)

# Sort by Drop-Column (gold standard)
comparison_norm = comparison_norm.reindex(drop_series.index)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
method_colors = ['#264653', '#2a9d8f', '#e9c46a']
titles = ['MDI (biased, fast)', 'Permutation (unbiased, slower)', 'Drop-Column (gold standard, slow)']

for ax, col, color, title in zip(axes, comparison_norm.columns, method_colors, titles):
    bar_colors = ['#2a9d8f' if f in real_features else '#e76f51' for f in comparison_norm.index]
    ax.barh(comparison_norm.index[::-1], comparison_norm[col][::-1],
            color=bar_colors[::-1], edgecolor='white')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Normalized Importance')
    ax.axvline(0, color='black', linewidth=0.5)

legend_handles = [
    mpatches.Patch(color='#2a9d8f', label='Real feature'),
    mpatches.Patch(color='#e76f51', label='Noise feature'),
]
axes[0].legend(handles=legend_handles, loc='lower right', fontsize=9)
fig.suptitle('Comparison of Feature Importance Methods\n(normalized for visual comparison)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nRank comparison (rank 1 = most important):")
rank_df = pd.DataFrame({
    'MDI_rank':        mdi_series.rank(ascending=False).astype(int),
    'Perm_rank':       perm_series.rank(ascending=False).astype(int),
    'DropCol_rank':    drop_series.rank(ascending=False).astype(int),
    'is_real':         [f in real_features for f in feature_names]
})
print(rank_df.sort_values('DropCol_rank'))

## The Bias Demo: High-Cardinality Noise

The `noise_float_hc` column is a **random uniform float in [0, 1000]** — it has absolutely no relationship with house price. Yet MDI often gives it a **surprisingly high importance** because:

1. A continuous float offers thousands of possible split thresholds
2. By pure chance, some of those splits happen to reduce impurity in a particular tree
3. MDI accumulates all of these "lucky" splits

**Permutation importance correctly exposes it** as near-zero or slightly negative, because shuffling a random column doesn't hurt (or help) predictions.

This is a real risk in practice: a high-cardinality identifier column (customer ID, transaction hash) can appear important by MDI even though it would cause terrible generalization.

In [ ]:
# ── Cell 9: Bias Demo — MDI vs Permutation for noise_float_hc ────────────────
noise_idx = feature_names.index('noise_float_hc')

mdi_noise_rank  = mdi_series.rank(ascending=False).astype(int)['noise_float_hc']
perm_noise_rank = perm_series.rank(ascending=False).astype(int)['noise_float_hc']

mdi_noise_val  = mdi_series['noise_float_hc']
perm_noise_val = perm_series['noise_float_hc']

print("=" * 55)
print("THE BIAS DEMO: noise_float_hc")
print("=" * 55)
print(f"  This feature is 100% random — zero true importance")
print(f"  MDI  importance : {mdi_noise_val:.4f}  (rank {mdi_noise_rank}/10)")
print(f"  Perm importance : {perm_noise_val:+.4f}  (rank {perm_noise_rank}/10)")
print()
if mdi_noise_rank <= 5:
    print("  *** MDI places this NOISE feature in the TOP HALF — biased! ***")
print(f"  Permutation correctly assigns it near-zero importance.")
print()

# Visual comparison just for these features
highlight_feats = ['sqft', 'location_score', 'school_rating', 'noise_float_hc', 'noise_binary_1']
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, (method, series, title) in zip(axes, [
    ('MDI', mdi_series[highlight_feats], 'MDI (biased)'),
    ('Permutation', perm_series[highlight_feats], 'Permutation (unbiased)')
]):
    bar_colors = ['#e76f51' if 'noise' in f else '#2a9d8f' for f in highlight_feats]
    ax.bar(highlight_feats, series.reindex(highlight_feats), color=bar_colors, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xticklabels(highlight_feats, rotation=30, ha='right')
    ax.set_ylabel('Importance')
    for i, (feat, val) in enumerate(series.reindex(highlight_feats).items()):
        ax.text(i, max(val + 0.002, 0.001), f'{val:.3f}', ha='center', fontsize=8)

fig.suptitle('MDI vs Permutation: The Bias Demo\n(noise_float_hc is 100% random)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 10: Importance Stability — Error Bars Over Multiple Seeds ────────────
# Train 10 forests with different seeds; record MDI and permutation for each.
# Shows: which method is more stable/reliable?

n_runs = 10
mdi_runs  = np.zeros((n_runs, len(feature_names)))
perm_runs = np.zeros((n_runs, len(feature_names)))

for i, seed in enumerate(range(n_runs)):
    rng_seed = seed * 7 + 13
    rf_i = RandomForestRegressor(
        n_estimators=100, min_samples_leaf=5, random_state=rng_seed, n_jobs=-1
    )
    rf_i.fit(X_train, y_train)
    mdi_runs[i] = rf_i.feature_importances_
    
    pi = permutation_importance(rf_i, X_val, y_val, n_repeats=5, random_state=rng_seed, scoring='r2')
    perm_runs[i] = pi.importances_mean

mdi_mean  = mdi_runs.mean(axis=0)
mdi_std   = mdi_runs.std(axis=0)
perm_mean = perm_runs.mean(axis=0)
perm_std  = perm_runs.std(axis=0)

# Sort by MDI mean for display
order = np.argsort(mdi_mean)[::-1]
feat_ordered = [feature_names[i] for i in order]
bar_colors   = ['#2a9d8f' if f in real_features else '#e76f51' for f in feat_ordered]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# MDI stability
axes[0].barh(feat_ordered[::-1], mdi_mean[order][::-1],
             xerr=mdi_std[order][::-1], color=bar_colors[::-1],
             edgecolor='white', capsize=4)
axes[0].set_title('MDI Importance\n(mean ± std over 10 forest seeds)', fontweight='bold')
axes[0].set_xlabel('MDI Importance')

# Permutation stability
perm_order = np.argsort(perm_mean)[::-1]
feat_perm_ordered = [feature_names[i] for i in perm_order]
bar_colors_perm   = ['#2a9d8f' if f in real_features else '#e76f51' for f in feat_perm_ordered]
axes[1].barh(feat_perm_ordered[::-1], perm_mean[perm_order][::-1],
             xerr=perm_std[perm_order][::-1], color=bar_colors_perm[::-1],
             edgecolor='white', capsize=4)
axes[1].set_title('Permutation Importance\n(mean ± std over 10 forest seeds)', fontweight='bold')
axes[1].set_xlabel('Permutation Importance (R² drop)')

legend_handles = [
    mpatches.Patch(color='#2a9d8f', label='Real feature'),
    mpatches.Patch(color='#e76f51', label='Noise feature'),
]
axes[0].legend(handles=legend_handles, loc='lower right')
plt.tight_layout()
plt.show()

# CoV = coefficient of variation = std/mean (lower = more stable)
print("\nCoefficient of Variation (std/|mean|) — lower = more stable:")
print(f"{'Feature':<22} {'MDI CoV':>10} {'Perm CoV':>10}")
print("-" * 45)
for i, feat in enumerate(feature_names):
    m_cov = mdi_std[i] / (abs(mdi_mean[i]) + 1e-9)
    p_cov = perm_std[i] / (abs(perm_mean[i]) + 1e-9)
    print(f"{feat:<22} {m_cov:>10.3f} {p_cov:>10.3f}")

In [ ]:
# ── Cell 11: SHAP Values ──────────────────────────────────────────────────────
# SHAP (SHapley Additive exPlanations) assigns each feature a contribution
# to each individual prediction, based on cooperative game theory.
#
# For a prediction on house i:
#   f(x_i) = E[f(X)] + SHAP_1(x_i) + SHAP_2(x_i) + ... + SHAP_p(x_i)
#
# Global importance: mean |SHAP| across all samples

try:
    import shap
    SHAP_AVAILABLE = True
    print(f"shap version: {shap.__version__} — running full SHAP analysis")
except ImportError:
    SHAP_AVAILABLE = False
    print("shap not installed. Install with: pip install shap")
    print("Showing conceptual demo instead.")

if SHAP_AVAILABLE:
    # TreeExplainer is optimized for tree-based models
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_val)

    # Global importance = mean |SHAP|
    shap_importance = np.abs(shap_values).mean(axis=0)
    shap_series = pd.Series(shap_importance, index=feature_names).sort_values(ascending=False)

    print(f"\nBaseline prediction (E[f(X)]): ${explainer.expected_value:,.0f}")
    print(f"\nSHAP Global Importance (mean |SHAP value|):")
    for rank, (feat, val) in enumerate(shap_series.items(), 1):
        tag = '(REAL)  ' if feat in real_features else '(NOISE) '
        print(f"  {rank:2d}. {tag}{feat:<22} ${val:,.0f}")

    # SHAP bar plot
    fig, ax = plt.subplots(figsize=(10, 5))
    bar_colors_shap = ['#2a9d8f' if f in real_features else '#e76f51' for f in shap_series.index]
    ax.barh(list(shap_series.index)[::-1], list(shap_series.values)[::-1],
            color=bar_colors_shap[::-1], edgecolor='white')
    ax.set_xlabel('Mean |SHAP value| (in $)')
    ax.set_title('SHAP Global Feature Importance\n(mean absolute SHAP values)', fontweight='bold')
    legend_handles = [
        mpatches.Patch(color='#2a9d8f', label='Real feature'),
        mpatches.Patch(color='#e76f51', label='Noise feature'),
    ]
    ax.legend(handles=legend_handles)
    plt.tight_layout()
    plt.show()

    # SHAP summary plot
    print("\nSHAP Summary Plot (beeswarm):")
    shap.summary_plot(shap_values, X_val, feature_names=feature_names, show=True)

else:
    # Manual SHAP-like concept: marginal contribution of sqft
    print("\n── Manual SHAP-like demo for sqft ──")
    print("Concept: SHAP(sqft, house_i) = avg over orderings of [f(features up to sqft) - f(features without sqft)]")
    print("We approximate this for 5 sample houses by comparing predictions with/without sqft perturbed:")
    
    sqft_idx = feature_names.index('sqft')
    baseline_pred = rf.predict(X_val[:5])
    X_zero_sqft = X_val[:5].copy()
    X_zero_sqft[:, sqft_idx] = X_val[:, sqft_idx].mean()  # replace with mean
    mean_pred = rf.predict(X_zero_sqft)
    
    print(f"\n{'House':>6} {'Full Pred':>12} {'Mean sqft Pred':>16} {'Contribution':>14}")
    print("-" * 52)
    for i in range(5):
        print(f"  {i+1:>4} ${baseline_pred[i]:>10,.0f} ${mean_pred[i]:>14,.0f} ${baseline_pred[i]-mean_pred[i]:>12,.0f}")

In [ ]:
# ── Cell 12: Final Summary Comparison ────────────────────────────────────────
print("=" * 65)
print("FEATURE IMPORTANCE METHOD COMPARISON")
print("=" * 65)

summary = pd.DataFrame({
    'MDI_rank':     mdi_series.rank(ascending=False).astype(int),
    'Perm_rank':    perm_series.rank(ascending=False).astype(int),
    'DropCol_rank': drop_series.rank(ascending=False).astype(int),
    'MDI_val':      mdi_series.round(4),
    'Perm_val':     perm_series.round(4),
    'DropCol_val':  drop_series.round(4),
    'is_real':      [f in real_features for f in feature_names]
}).sort_values('DropCol_rank')

print(summary.to_string())

noise_float_mdi_rank  = summary.loc['noise_float_hc', 'MDI_rank']
noise_float_perm_rank = summary.loc['noise_float_hc', 'Perm_rank']
noise_float_dc_rank   = summary.loc['noise_float_hc', 'DropCol_rank']

print(f"""
Key Takeaways:
  noise_float_hc (100% random) ranks:
    MDI:       {noise_float_mdi_rank}/10  ← bias toward high cardinality
    Permutation: {noise_float_perm_rank}/10  ← correctly near bottom
    Drop-Column: {noise_float_dc_rank}/10  ← correctly near bottom

  Use MDI for: quick exploration, relative ordering of features (be cautious with high-cardinality)
  Use Permutation for: production feature selection, model-agnostic comparisons
  Use Drop-Column for: final validation when compute budget allows
  Use SHAP for: explaining individual predictions and global ranking
""")

## Method Summary Table

| Method | Speed | Bias? | Model-agnostic? | Uses validation data? | Best use |
|---|---|---|---|---|---|
| **MDI** | ⚡ Very fast | Yes (high-cardinality) | No (trees only) | No (training) | Quick exploration |
| **Permutation** | Medium | No | Yes | Yes | Feature selection, reporting |
| **Drop-Column** | Slow (N refits) | No | Yes | Yes | Final validation |
| **SHAP** | Medium-slow | No | Model-dependent | Optional | Individual + global explanations |

### When MDI and Permutation Disagree
- **MDI high, Permutation low:** Likely high-cardinality feature exploited by MDI (noise)
- **MDI low, Permutation high:** Rare; can happen if feature only matters near leaves
- **Both high:** Feature is genuinely important
- **Both disagree for correlated features:** Neither captures true importance of either feature alone

## Self-Check Questions

Test your understanding before moving on.

---

### Question 1
MDI gives a random float column importance = 0.12 (4th highest out of 10 features). Permutation gives it importance = −0.001. Which should you trust and why?

<details>
<summary>Answer</summary>

**Trust the permutation importance.** 

MDI is biased toward high-cardinality features. A random float column has thousands of possible split thresholds, and by pure chance some of those splits reduce impurity in specific trees. MDI accumulates all these "lucky" splits, inflating the apparent importance. 

Permutation importance is not fooled by this: shuffling a truly random column doesn't change predictions (since the model's "use" of that column was spurious), so the score drop is near zero. The slightly negative value (−0.001) is just random fluctuation from the shuffling process.

**Practical rule:** Any time MDI ranks a feature highly but permutation doesn't, check the feature's cardinality. If it has many unique values, be suspicious.

</details>

---

### Question 2
You have two features with correlation = 0.95 (e.g., `total_rooms` and `sqft`). What will happen to their permutation importances? Is either one "truly" more important?

<details>
<summary>Answer</summary>

**Both features will appear less important than they truly are.** 

When you shuffle `total_rooms`, the model can still "cheat" by using `sqft` (which is highly correlated and unchanged). So the performance drop is small — making `total_rooms` look unimportant. The same happens when you shuffle `sqft`. Result: both appear weak even though together they contain the true signal.

**Is either truly more important?** This is a genuinely difficult question. In the presence of near-perfect correlation, neither permutation nor drop-column importance can cleanly separate the two. The right approach is:
1. Use domain knowledge to choose one
2. Or use a grouped permutation importance (shuffle both simultaneously)
3. Or use SHAP with a model that handles correlation explicitly

The correlation issue is one reason feature importance should never be the sole input to feature selection decisions.

</details>

---

### Question 3
Permutation importance is computed on the validation set, not training set. Why does this matter?

<details>
<summary>Answer</summary>

**Because training-set importance reflects memorization, not generalization.**

A Random Forest can overfit to certain features on the training set — particularly high-cardinality features that allow near-perfect memorization. If you computed permutation importance on training data, shuffling that noisy feature would still hurt performance (because the model memorized it). This would incorrectly mark it as important.

On the **validation set**, the model's spurious memorization doesn't transfer. Shuffling a noise feature has no effect because the model's exploitation of that feature was a training artifact.

More broadly: validation-set importance measures what the model **generalizes by**, which is exactly what we care about for deployment. Training-set importance measures what the model **memorized by**, which is much less useful.

</details>

In [ ]:
# ── Cell 13: Bonus — Partial Dependence Plot for top feature ──────────────────
# Partial dependence shows how the predicted price changes as one feature varies,
# marginalizing over all other features.

top_feature_by_perm = perm_series.index[0]  # typically 'sqft' or 'location_score'
top_feat_idx = feature_names.index(top_feature_by_perm)

print(f"Generating Partial Dependence Plot for: {top_feature_by_perm}")

# Create a grid of values for the top feature
feature_grid = np.linspace(X_val[:, top_feat_idx].min(), X_val[:, top_feat_idx].max(), 50)

# For each grid value, set that feature for all val samples and average predictions
pdp_means = []
pdp_stds  = []
for val_f in feature_grid:
    X_pdp = X_val.copy()
    X_pdp[:, top_feat_idx] = val_f
    preds = rf.predict(X_pdp)
    pdp_means.append(preds.mean())
    pdp_stds.append(preds.std())

pdp_means = np.array(pdp_means)
pdp_stds  = np.array(pdp_stds)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(feature_grid, pdp_means, color='#2a9d8f', linewidth=2.5, label='Partial Dependence')
ax.fill_between(feature_grid, pdp_means - pdp_stds, pdp_means + pdp_stds,
                alpha=0.2, color='#2a9d8f', label='±1 std (ICE spread)')
ax.set_xlabel(top_feature_by_perm)
ax.set_ylabel('Predicted House Price ($)')
ax.set_title(f'Partial Dependence Plot: {top_feature_by_perm}\n(average prediction as feature varies, others held at their values)', fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

print(f"As {top_feature_by_perm} increases from {feature_grid[0]:.1f} to {feature_grid[-1]:.1f},")
print(f"  predicted price changes by ${pdp_means[-1] - pdp_means[0]:+,.0f}")

## Key Takeaways

1. **MDI is fast but biased** — always verify with permutation importance before reporting results. High-cardinality noise features can look deceptively important.

2. **Permutation importance is the practical standard** — model-agnostic, unbiased, uses validation data. Use it for feature selection decisions.

3. **Drop-column is the gold standard** — but costs N extra model fits. Reserve for final validation.

4. **Correlated features dilute each other's importance** — neither method cleanly separates them. Domain knowledge or grouped importance is needed.

5. **Importance ≠ causation** — a feature can appear important because it is correlated with a truly causal variable.

6. **SHAP connects local (individual predictions) and global (overall importance)** — the only method that tells you both "which features matter overall" and "why did the model predict $420K for *this specific house*".

---

**Next:** Notebook 8 — AdaBoost: Sequential Boosting with Adaptive Weights